In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [70]:
df = pd.read_csv('weekly_driving_profiles.csv')

In [ ]:
cols = ['clear_weather','weather_wind_speed_mean','weather_visibility_mean','forward_collision', 'distracted_driver','too_close_distance','lane_departure','driver_making_calls','driver_smoking','fatigue_driving']

df = df.drop(columns=cols)

In [71]:
import os
import wandb # для логирования

import numpy as np
import random
from tqdm import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim # для оптимизаторов
from torchvision import datasets # для данных
import torchvision.transforms as transforms # для преобразований тензоров


In [72]:
class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']
  input_dim = len(feature_cols)
  hidden_dims = [32, 16, 8]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60

hidden_dims = [32, 16, 8] - была совершена проверка (сравнение) разного количества нейронов на 1 слое

Каждый следующий шаг (слой) обобщает информацию (уменьшение нейронов), находии общие зависимости

Adam оптимизатор (по умолчанию) работает лучше всего с lr=0.001 - базовый выбор

60 эпох - практическая проверка. Лучшее обучение на более ранней эпохе, дальше уже переобучение начинается



In [73]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']
target_col = 'claims_count'

unique_drivers = df['driver_id'].unique()

#примерно 70% обучение 30% тест
train_drivers, test_drivers = train_test_split(unique_drivers, test_size=0.3, random_state=30)

X_train = df[df['driver_id'].isin(train_drivers)][feature_cols]
X_test = df[df['driver_id'].isin(test_drivers)][feature_cols]
y_train = df[df['driver_id'].isin(train_drivers)][target_col]
y_test = df[df['driver_id'].isin(test_drivers)][target_col]

print("Количество водителей в тренировке: ",len(train_drivers), "Количество строк: ",len(X_train))
print("Количество водителей в тесте: ",len(test_drivers), "Количество строк: ",len(X_test))


#масштабирование признаков
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)


train_data = TensorDataset(X_train_t, y_train_t)
test_data = TensorDataset(X_test_t, y_test_t)

#рекомендуют для датасета 10к-100к батчсайз 64
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)


Количество водителей в тренировке:  247 Количество строк:  8465
Количество водителей в тесте:  107 Количество строк:  4063


In [74]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток
seed_everything(30)

In [75]:
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x

3 слоя Linear - базовая архитектура

ReLU() - базовая функция активации

torch.exp(self.net(x)) - экспонента для положительного результата

In [76]:
model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель


Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=8, bias=True)
    (5): ReLU()
    (6): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [77]:
# функция потерь
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

Для задачи регрессии, где нам важно больше штрафовать за большие ошибки, чем за маленькие хорошо подходит MSE

Adam - базовый хороший оптимизатор


In [78]:
# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0


    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)

    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(
        train_loss_sum / len(train_loader.dataset)))



In [79]:
# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    tqdm.write('Test set: Average loss: {:.4f}'.format(
       test_loss_sum / len(test_loader.dataset)))
    return test_loss_sum / len(test_loader.dataset)



In [80]:
# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    model = model.to(device)

    losses = []
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch)
        losses.append(test(model, device, test_loader, criterion))
    print('Training is ended!')
    best_epoch = np.argmin(losses)+1
    print()
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',min(losses))


In [81]:
main(model)



Epoch: 1


100%|██████████| 133/133 [00:00<00:00, 353.65it/s]



Train set: Average loss: 0.9617
Test set: Average loss: 0.7692

Epoch: 2


100%|██████████| 133/133 [00:00<00:00, 476.40it/s]



Train set: Average loss: 0.7187
Test set: Average loss: 0.6316

Epoch: 3


...


Train set: Average loss: 0.1216
Test set: Average loss: 0.1820

Epoch: 59


100%|██████████| 133/133 [00:00<00:00, 432.23it/s]



Train set: Average loss: 0.1207
Test set: Average loss: 0.1940

Epoch: 60


100%|██████████| 133/133 [00:00<00:00, 467.96it/s]



Train set: Average loss: 0.1194
Test set: Average loss: 0.1890
Training is ended!

BEST EPOCH:  42 with Loss:  0.1681321774548185


Реализована базовая модель полносвязной нейронной сети

Проверим гиперпараметры, выберем лучшие варианты

1. Количество нейронов на 1 слое

варианты:

X - количество параметров - 19шт
Нужна убывающая регрессия (для уменьшения количества нейронов на каждом последующем слое), идем по степеням двойки

X*1 - 19, берем 32 нейрона в базовом слое [32, 16, 8]

X*2 - 38, берем 64 нейрона в базовом слое [64, 32, 16]

X*4 - 76, берем 128 нейронов в базовом слое [128, 64, 32]

X*7 - 133, берем 258 нейронов в базовом слое [256, 128, 64]

X*8 - 152, дальше нет смысла идти, высокий шанс переобучения

In [82]:
class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

  input_dim = len(feature_cols)
  hidden_dims =  [128, 64, 32]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x
model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель

# функция потерь
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0


    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)

    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(
        train_loss_sum / len(train_loader.dataset)))
# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    tqdm.write('Test set: Average loss: {:.4f}'.format(
       test_loss_sum / len(test_loader.dataset)))
    return test_loss_sum / len(test_loader.dataset)

# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    model = model.to(device)

    losses = []
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch)
        losses.append(test(model, device, test_loader, criterion))
    print('Training is ended!')
    best_epoch = np.argmin(losses)+1
    print()
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',min(losses))



Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [83]:
main(model)


Epoch: 1


100%|██████████| 133/133 [00:00<00:00, 370.16it/s]



Train set: Average loss: 0.2179
Test set: Average loss: 0.1726

Epoch: 2


100%|██████████| 133/133 [00:00<00:00, 278.40it/s]



Train set: Average loss: 0.1404
Test set: Average loss: 0.1698

Epoch: 3


...


Train set: Average loss: 0.0354
Test set: Average loss: 0.2447

Epoch: 59


100%|██████████| 133/133 [00:00<00:00, 408.61it/s]



Train set: Average loss: 0.0327
Test set: Average loss: 0.2385

Epoch: 60


100%|██████████| 133/133 [00:00<00:00, 412.27it/s]


Train set: Average loss: 0.0357
Test set: Average loss: 0.2629
Training is ended!

BEST EPOCH:  2 with Loss:  0.16978198520100207


In [84]:
class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

  input_dim = len(feature_cols)
  hidden_dims =   [64, 32, 16]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x
model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель

# функция потерь
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0


    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)

    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(
        train_loss_sum / len(train_loader.dataset)))
# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    tqdm.write('Test set: Average loss: {:.4f}'.format(
       test_loss_sum / len(test_loader.dataset)))
    return test_loss_sum / len(test_loader.dataset)

# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    model = model.to(device)

    losses = []
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch)
        losses.append(test(model, device, test_loader, criterion))
    print('Training is ended!')
    best_epoch = np.argmin(losses)+1
    print()
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',min(losses))



Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [85]:
main(model)


Epoch: 1


100%|██████████| 133/133 [00:00<00:00, 487.98it/s]



Train set: Average loss: 0.2092
Test set: Average loss: 0.1726

Epoch: 2


100%|██████████| 133/133 [00:00<00:00, 476.04it/s]



Train set: Average loss: 0.1400
Test set: Average loss: 0.1705

Epoch: 3


...


Train set: Average loss: 0.0616
Test set: Average loss: 0.2265

Epoch: 59


100%|██████████| 133/133 [00:00<00:00, 403.67it/s]



Train set: Average loss: 0.0600
Test set: Average loss: 0.2343

Epoch: 60


100%|██████████| 133/133 [00:00<00:00, 450.30it/s]



Train set: Average loss: 0.0611
Test set: Average loss: 0.2100
Training is ended!

BEST EPOCH:  3 with Loss:  0.17034084483231124


In [86]:
class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

  input_dim = len(feature_cols)
  hidden_dims =   [256, 128, 64]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x
model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель

# функция потерь
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0


    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)

    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(
        train_loss_sum / len(train_loader.dataset)))
# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    tqdm.write('Test set: Average loss: {:.4f}'.format(
       test_loss_sum / len(test_loader.dataset)))
    return test_loss_sum / len(test_loader.dataset)

# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    model = model.to(device)

    losses = []
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch)
        losses.append(test(model, device, test_loader, criterion))
    print('Training is ended!')
    best_epoch = np.argmin(losses)+1
    print()
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',min(losses))



Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [87]:
main(model)


Epoch: 1


100%|██████████| 133/133 [00:00<00:00, 340.01it/s]



Train set: Average loss: 0.1677
Test set: Average loss: 0.1710

Epoch: 2


100%|██████████| 133/133 [00:00<00:00, 335.25it/s]



Train set: Average loss: 0.1345
Test set: Average loss: 0.1714

Epoch: 3


...


Train set: Average loss: 0.0319
Test set: Average loss: 0.2863

Epoch: 59


100%|██████████| 133/133 [00:00<00:00, 292.81it/s]



Train set: Average loss: 0.0278
Test set: Average loss: 0.2789

Epoch: 60


100%|██████████| 133/133 [00:00<00:00, 307.44it/s]


Train set: Average loss: 0.0341
Test set: Average loss: 0.2497
Training is ended!

BEST EPOCH:  1 with Loss:  0.17096189426456712


Вывод:

32 нейрона:
BEST EPOCH:  42 with Loss:  0.1681321774548185

64 нейрона:
BEST EPOCH:  3 with Loss:  0.17034084483231124

128 нейронов:
BEST EPOCH:  2 with Loss:  0.16978198520100207

256 нейронов:
BEST EPOCH:  1 with Loss:  0.17096189426456712

Лучший результат у 32 нейронов, но в целом нет сильных отличий между количеством нейронов. При этом можно отметить скорость обучения - 32 нейрона лучше всего показали себя только на 42 эпохе, а 64/128/256 на 3/2/1 эпохе, что является показателем недо/переобучения